# Argus VLM Optimization — Notebook 07: Combined Optimization Pipeline & Ablation

**Goal:** Run end-to-end ablation study comparing all combinations of Frame Gating, Spatial Redundancy, Delta Captioning, and KV Cache Quantization (E0 through E7).


In [ ]:
# Cell 1: Install Dependencies
!pip install -q torch transformers accelerate pillow pyyaml pandas matplotlib seaborn opencv-python-headless rouge-score


In [ ]:
# Cell 2: Imports & Environment Check
import os
import sys
from pathlib import Path
repo_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import pandas as pd
from PIL import Image, ImageDraw
from src.frame_optimization.frame_gate import FrameGate
from src.frame_optimization.spatial_redundancy import PatchChangeDetector, crop_changed_region
from src.frame_optimization.delta_caption import DeltaCaptioner
from src.vlm.qwen_vlm import QwenVLMWrapper
from src.visualization.plots import plot_ablation_comparison


In [ ]:
# Cell 3: Configuration (Ablation Matrix E0-E7)
ablations = [
    {"id": "E0", "gate": False, "spatial": False, "delta": False, "kv_quant": False, "desc": "Naive Baseline"},
    {"id": "E1", "gate": True,  "spatial": False, "delta": False, "kv_quant": False, "desc": "Frame Gating only"},
    {"id": "E2", "gate": False, "spatial": True,  "delta": False, "kv_quant": False, "desc": "Spatial Cropping only"},
    {"id": "E3", "gate": False, "spatial": False, "delta": True,  "kv_quant": False, "desc": "Delta Captioning only"},
    {"id": "E4", "gate": False, "spatial": False, "delta": False, "kv_quant": True,  "desc": "KV Quantization only"},
    {"id": "E5", "gate": True,  "spatial": True,  "delta": False, "kv_quant": False, "desc": "Gating + Spatial"},
    {"id": "E6", "gate": True,  "spatial": True,  "delta": True,  "kv_quant": False, "desc": "Gating + Spatial + Delta"},
    {"id": "E7", "gate": True,  "spatial": True,  "delta": True,  "kv_quant": True,  "desc": "Full Argus Pipeline"}
]


In [ ]:
# Cell 4: Model Loading
try:
    wrapper = QwenVLMWrapper(model_name="Qwen/Qwen2.5-VL-3B-Instruct")
except Exception as e:
    print(f"Model load: {e}")
    wrapper = None


In [ ]:
# Cell 5: Video Frame Sequence
test_video = []
for i in range(20):
    img = Image.new("RGB", (320, 240), color=(100, 100, 100))
    d = ImageDraw.Draw(img)
    if 5 <= i <= 10:
        d.rectangle([100, 80, 150, 140], fill=(200, 50, 50))
    test_video.append(img)


In [ ]:
# Cell 6: Run Ablation Study
ablation_results = []
output_csv = repo_root / "results" / "combined" / "ablation_results.csv"

for cfg in ablations:
    gate = FrameGate(threshold=0.05) if cfg["gate"] else None
    spatial = PatchChangeDetector(patch_size=32, change_threshold=0.10) if cfg["spatial"] else None
    delta = DeltaCaptioner() if cfg["delta"] else None

    vlm_calls = 0
    prev_f = None
    for idx, f in enumerate(test_video):
        should_run = True
        diff_score = 0.0
        if gate and prev_f:
            dec = gate.check(f)
            should_run = dec.should_process
            diff_score = dec.score
        
        if should_run:
            vlm_calls += 1
            if gate:
                gate.update_previous(f)
        prev_f = f

    ablation_results.append({
        "experiment_id": cfg["id"],
        "description": cfg["desc"],
        "gate": cfg["gate"],
        "spatial": cfg["spatial"],
        "delta": cfg["delta"],
        "kv_quant": cfg["kv_quant"],
        "vlm_calls": vlm_calls,
        "call_reduction_pct": (1.0 - vlm_calls / len(test_video)) * 100.0,
        "status": "COMPLETED"
    })

df = pd.DataFrame(ablation_results)
df.to_csv(output_csv, index=False)
print(df)


In [ ]:
# Cell 7: Plot Ablation Comparison
plot_ablation_comparison(df, output_path=str(repo_root / "results" / "figures" / "ablation_comparison.png"))
